In [3]:
import onnxruntime as ort
import cv2
import numpy as np
import os
import torch
import torchvision

# --- Configuration ---
model_path = "models/best_ckpt.onnx"
image_path = (
    "datasets/your_dataset/val2017/0-2_jpg.rf.47ba42c1c3ebf77468a8610a9ff5e828.jpg"
)
input_shape = (
    320,
    320,
)  # Expected H, W for the model after batch and channel dimensions
input_dtype = np.float32  # Expected data type for the model input

# --- Verify file paths ---
if not os.path.exists(model_path):
    print(f"Error: Model file not found at '{model_path}'")
    print(
        "Please ensure the 'models' directory exists in the same location as this script and contains 'best_ckpt.onnx'."
    )
    exit()

if not os.path.exists(image_path):
    print(f"Error: Image file not found at '{image_path}'")
    print(
        "Please ensure the 'datasets/your_dataset/val2017' directory exists and contains '0-2_jpg.rf.47ba42c1c3ebf77468a8610a9ff5e828.jpg'."
    )
    exit()

# --- 1. Load the ONNX model ---
print(f"Loading ONNX model from: {model_path}")
try:
    # Create an inference session with the ONNX model
    session = ort.InferenceSession(model_path, providers=ort.get_available_providers())
    # Get input and output names from the model
    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name
    print(
        f"Model loaded successfully. Input name: '{input_name}', Output name: '{output_name}'"
    )
except Exception as e:
    print(f"Error loading ONNX model: {e}")
    exit()

# --- 2. Load and preprocess the image ---
print(f"Loading and preprocessing image from: {image_path}")
try:
    # Read the image using OpenCV
    original_image = cv2.imread(image_path)
    if original_image is None:
        raise ValueError(
            "Image could not be loaded. Check if the path is correct and the image is valid."
        )

    # Convert BGR to RGB (OpenCV loads images as BGR by default)
    image_rgb = cv2.cvtColor(original_image, cv2.COLOR_BGR2RGB)

    # Resize the image to the model's expected input dimensions
    # INTER_LINEAR is a good default for resizing
    resized_image = cv2.resize(image_rgb, input_shape, interpolation=cv2.INTER_LINEAR)

    # Normalize pixel values to [0, 1]
    # Models often expect input in this range, or scaled by 255.0
    normalized_image = resized_image.astype(input_dtype) / 255.0

    # Transpose the image from HWC (Height, Width, Channel) to CHW (Channel, Height, Width)
    # The model expects [1, 3, 320, 320] which means C, H, W
    transposed_image = np.transpose(
        normalized_image, (2, 0, 1)
    )  # (H, W, C) -> (C, H, W)

    # Add a batch dimension: (C, H, W) -> (1, C, H, W)
    # The model expects a batch size of 1
    input_tensor = np.expand_dims(transposed_image, axis=0)

    print(
        f"Image preprocessed. Input tensor shape: {input_tensor.shape}, dtype: {input_tensor.dtype}"
    )

except Exception as e:
    print(f"Error during image preprocessing: {e}")
    exit()


def postprocess(
    prediction, num_classes, conf_thre=0.7, nms_thre=0.45, class_agnostic=False
):
    # new box_corner [1, 2100, 4] use numpy
    box_corner = np.zeros_like(prediction[:, :, :4])
    box_corner[:, :, 0] = prediction[:, :, 0] - prediction[:, :, 2] / 2
    box_corner[:, :, 1] = prediction[:, :, 1] - prediction[:, :, 3] / 2
    box_corner[:, :, 2] = prediction[:, :, 0] + prediction[:, :, 2] / 2
    box_corner[:, :, 3] = prediction[:, :, 1] + prediction[:, :, 3] / 2
    prediction[:, :, :4] = box_corner[:, :, :4]

    print("prediction:", prediction[0][0])
    print("prediction:", prediction[0][1])
    print("prediction:", prediction[0][2])

    output = [None for _ in range(len(prediction))]
    for i, image_pred in enumerate(prediction):
        print("image_pred shape:", image_pred.shape)
        
        image_pred = torch.from_numpy(image_pred)

        # If none are remaining => process next image
        if not image_pred.size(0):
            continue
        # Get score and class with highest confidence
        class_conf, class_pred = torch.max(
            image_pred[:, 5 : 5 + num_classes], 1, keepdim=True
        )
        print("image_pred:", image_pred[0, 5 : 5 + num_classes])
        print("class_conf shape:", class_conf.shape)
        print("class_conf sample:", class_conf[0])
        print("class_conf sample:", class_conf[1])
        print("class_pred shape:", class_pred.shape)
        print("class_pred sample:", class_pred[0])
        print("class_pred sample:", class_pred[1])

        print("class_conf.squeeze(): ", class_conf.squeeze())

        conf_mask = (image_pred[:, 4] * class_conf.squeeze() >= conf_thre).squeeze()
        print("conf_mask shape:", conf_mask.shape)
        print("conf_mask:", conf_mask)
        print("conf_mask True:", torch.sum(conf_mask).item())

        # Detections ordered as (x1, y1, x2, y2, obj_conf, class_conf, class_pred)
        detections = torch.cat((image_pred[:, :5], class_conf, class_pred.float()), 1)
        print("detections sample:", detections[2000])
        detections = detections[conf_mask]
        print("detections shape:", detections.shape)
        print("detections:", detections)
        if not detections.size(0):
            continue

        if class_agnostic:
            nms_out_index = torchvision.ops.nms(
                detections[:, :4],
                detections[:, 4] * detections[:, 5],
                nms_thre,
            )
        else:
            nms_out_index = torchvision.ops.batched_nms(
                detections[:, :4],
                detections[:, 4] * detections[:, 5],
                detections[:, 6],
                nms_thre,
            )

        detections = detections[nms_out_index]
        if output[i] is None:
            output[i] = detections
        else:
            output[i] = torch.cat((output[i], detections))

    print("Final output shape:", len(output), "tensors")
    print("Output example:", output[0] if output else "No detections")
    return output


# --- 3. Perform inference ---
print("Performing inference...")
# Run the model inference
# The run method takes a list of output names and a dictionary of inputs
outputs = session.run([output_name], {input_name: input_tensor})

# The output is a list, so we take the first element
model_output = outputs[0]
print("Inference completed successfully!")
print(f"Output shape: {model_output.shape}")
print(f"Output data type: {model_output.dtype}")

# --- 4. Handle the output ---
# The output shape is [1, 2100, 7]
# This typically means:
#   - Batch size: 1
#   - Number of detections/proposals: 2100
#   - Features per detection (e.g., [x, y, w, h, confidence, class1_prob, class2_prob, ...])
#
# Let's print the first 5 detections (or fewer if 2100 is not met)
num_detections_to_show = min(5, model_output.shape[1])
print(f"\nFirst {num_detections_to_show} entries of the output (sample):")
for i in range(num_detections_to_show):
    print(f"Detection {i+1}: {model_output[0, i, :]}")
    
# You would typically add post-processing here, such as:
# - Non-Maximum Suppression (NMS) to filter overlapping bounding boxes
# - Thresholding on confidence scores
# - Mapping class IDs to actual class names

processed_outputs = postprocess(
    model_output,
    2,
    0.25,
    0.45,
    class_agnostic=True,
)

print("\nScript finished.")

Loading ONNX model from: models/best_ckpt.onnx
Model loaded successfully. Input name: 'images', Output name: 'output'
Loading and preprocessing image from: datasets/your_dataset/val2017/0-2_jpg.rf.47ba42c1c3ebf77468a8610a9ff5e828.jpg
Image preprocessed. Input tensor shape: (1, 3, 320, 320), dtype: float32
Performing inference...
Inference completed successfully!
Output shape: (1, 2100, 7)
Output data type: float32

First 5 entries of the output (sample):
Detection 1: [ 2.7010953e-01  3.6748089e-03 -3.5661459e-04  1.6869573e-01
  2.8908253e-06  1.0443926e-02  1.0016948e-02]
Detection 2: [ 2.8186211e-01 -1.8817816e-02  2.4213735e-02  1.7995226e-01
  9.2387199e-07  1.0425836e-02  9.8882914e-03]
Detection 3: [ 2.7802950e-01 -3.0333074e-02  3.0614330e-02  1.9168875e-01
  1.1622906e-06  1.0267824e-02  9.8963678e-03]
Detection 4: [ 2.7085590e-01 -3.4567311e-02  3.0283704e-02  1.9364940e-01
  1.1622906e-06  1.0242015e-02  9.9039078e-03]
Detection 5: [ 2.6571828e-01 -4.1269325e-02  2.9018112e-0

In [39]:
!python demo/ONNXRuntime/onnx_inference.py -m models/best_ckpt.onnx -i datasets/your_dataset/val2017/0-2_jpg.rf.47ba42c1c3ebf77468a8610a9ff5e828.jpg -o YOLOX_outputs/vis_res -s 0.3 --input_shape 320,320

Input shape: (320, 320)
Original image shape: (320, 320, 3)
Original image sample: [172 169 148]
Input image shape: (320, 320, 3)
Input image sample: [172. 169. 148.]
Input image ratio: 1.0
Output shape: (1, 2100, 7)
Output sample: [4.30623174e-01 1.97660789e-01 1.31668001e-02 1.24691606e-01
 3.30805779e-06 1.12398863e-02 1.16201937e-02]
Postprocessing outputs with img_size: (320, 320) and p6: False
HSizes: [40, 20, 10]
WSizes: [40, 20, 10]
Grids shape: (1, 2100, 2)
Grids sample: [0 0]
Expanded strides shape: (1, 2100, 1)
Expanded strides sample: [8]
Predictions shape: (2100, 7)
Predictions sample: [3.4449854e+00 1.5812863e+00 8.1060305e+00 9.0623922e+00 3.3080578e-06
 1.1239886e-02 1.1620194e-02]
predictions[:, 4:5]:  [[3.3080578e-06]
 [6.2584877e-07]
 [1.0132790e-06]
 ...
 [7.8678131e-06]
 [7.8082085e-06]
 [1.9609928e-05]]
Boxes shape: (2100, 4)
Scores shape: (2100, 2)
Scores sample: [3.7182193e-08 3.8440273e-08]
Boxes shape: (2100, 4)
Boxes sample: [-0.60802984 -2.9499097   7.498000